# 11 — Inference and Agent Integration
Load checkpoint, run snapshot inference, and produce integration-ready prediction records.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from gat_traffic.predictor import Predictor

PROJECT_ROOT = Path("..").resolve()
ckpt = PROJECT_ROOT / "artifacts" / "checkpoints" / "gat_v1.pt"
df = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "synthetic_snapshots.csv")
feat_cols = ["current_speed_kmh", "current_density", "queue_length", "capacity_factor"]


In [ ]:
if ckpt.exists():
    pred = Predictor.from_checkpoint(ckpt)
    snapshot_id = int(df.snapshot_id.max())
    sdf = df[df.snapshot_id == snapshot_id].sort_values("road_segment_id")
    records = pred.predict(
        snapshot_id=snapshot_id,
        prediction_horizon_sec=300,
        road_segment_ids=sdf.road_segment_id.tolist(),
        features=sdf[feat_cols].to_numpy(dtype=np.float32),
    )
    out = pd.DataFrame([r.__dict__ for r in records])
    out_path = PROJECT_ROOT / "artifacts" / "metrics" / "inference_output_sample.csv"
    out.to_csv(out_path, index=False)
    print(out.head())
    print("saved", out_path)
else:
    print("Checkpoint not found. Run Notebook 08 training first.")


## Integration interface for future agent
Output fields:
- `snapshot_id`
- `prediction_horizon_sec`
- `road_segment_id`
- `predicted_travel_time_sec`
- `model_version`
- `is_valid`

Downstream routing logic must remain separate from this predictor.


In [ ]:
print("This notebook demonstrates prediction serving, not route optimization.")
